In [ ]:
# # N-RDD2024 Dataset Analysis
# Statistical analysis of the N-RDD2024 road damage dataset (YOLO txt format).
# Covers class distribution, bbox geometry, spatial density, annotation load,
# class co-occurrence, and COCO size categories across all countries.


In [ ]:
import json
import yaml
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_ROOT = Path(
    "/kaggle/input/datasets/sannyshankaranml/n-rdd2024"
    "/N-RDD2024Road damage and defects"
    "/Training and Validation Dataset"
)
OUTPUT_DIR = Path("/kaggle/working/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Classes (from data.yaml) ───────────────────────────────────────────────
CLASS_NAMES = ['D00', 'D10', 'D20', 'D30', 'D40',
               'D50', 'D60', 'D70', 'D80', 'D90']
NUM_CLASSES = len(CLASS_NAMES)

# ── Colors ─────────────────────────────────────────────────────────────────
CLASS_COLORS = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))
FIGURE_DPI = 120
plt.rcParams.update({"figure.dpi": FIGURE_DPI, "font.size": 10})

print(f"Dataset root: {DATASET_ROOT}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")


## 1. Discover country / split combinations


In [ ]:
COUNTRIES = [
    "Czech Republic_txt",
    "USA_txt",
    "china-motorbike_txt",
    "india_txt",
]

splits_found = {}
for country in COUNTRIES:
    for split in ["train", "valid"]:
        labels_dir = DATASET_ROOT / country / split / "labels"
        if labels_dir.exists():
            n = len(list(labels_dir.glob("*.txt")))
            splits_found.setdefault(country, []).append(split)
            print(f"  {country}/{split}/labels  →  {n:,} label files")


## 2. Parse all YOLO label files


In [ ]:
annotations = []   # list of dicts per bounding box

for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for lf in labels_dir.glob("*.txt"):
            with open(lf) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cid = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:5])
                    if w <= 0 or h <= 0 or cid >= NUM_CLASSES:
                        continue
                    perim = 2 * (w + h)
                    annotations.append({
                        "class_id":   cid,
                        "cx": cx, "cy": cy,
                        "w": w,   "h": h,
                        "area":        w * h,
                        "aspect":      w / h,
                        "compactness": (4 * np.pi * w * h) / perim**2 if perim > 0 else 0,
                        "country":     country.replace("_txt", ""),
                        "split":       split,
                        "img_stem":    lf.stem,
                    })

print(f"Total annotations: {len(annotations):,}")

# Build per-class numpy arrays
arrays = {i: defaultdict(list) for i in range(NUM_CLASSES)}
for ann in annotations:
    for key in ["cx", "cy", "w", "h", "area", "aspect", "compactness"]:
        arrays[ann["class_id"]][key].append(ann[key])
arrays = {cid: {k: np.array(v) for k, v in d.items()} for cid, d in arrays.items()}


## 3. Summary statistics


In [ ]:
print(f"\n{'Class':<8} {'Count':>8}  {'Mean W':>8}  {'Mean H':>8}  {'Mean Area':>10}")
print("-" * 55)
for cid, name in enumerate(CLASS_NAMES):
    d = arrays[cid]
    n = len(d.get("w", []))
    if n == 0:
        print(f"  {name:<6} {'0':>8}")
        continue
    print(f"  {name:<6} {n:>8,}  {np.mean(d['w']):>8.4f}  "
          f"{np.mean(d['h']):>8.4f}  {np.mean(d['area']):>10.6f}")


## Plot 01 — Class Instance Counts


In [ ]:
counts = [len(arrays[i].get("w", [])) for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(CLASS_NAMES, counts, color=CLASS_COLORS, edgecolor="white")
ax.bar_label(bars, labels=[f"{c:,}" for c in counts], padding=6, fontsize=10, fontweight="bold")
ax.set_xlabel("Annotated instances")
ax.set_title("N-RDD2024 — Class Instance Counts", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, max(counts) * 1.18 if max(counts) > 0 else 1)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_class_instance_counts.png", bbox_inches="tight")
plt.show()


## Plot 02 — Class Share Pie


In [ ]:
nz_idx    = [i for i, c in enumerate(counts) if c > 0]
nz_names  = [CLASS_NAMES[i] for i in nz_idx]
nz_counts = [counts[i]      for i in nz_idx]
nz_colors = [CLASS_COLORS[i] for i in nz_idx]

fig, ax = plt.subplots(figsize=(7, 6))
wedges, texts, autotexts = ax.pie(
    nz_counts, labels=nz_names, colors=nz_colors,
    autopct="%1.1f%%", startangle=140, pctdistance=0.82,
    wedgeprops={"edgecolor": "white", "linewidth": 1.2},
)
for t in autotexts:
    t.set_fontsize(9); t.set_fontweight("bold")
ax.set_title(f"Class Distribution  (total: {sum(nz_counts):,} instances)",
             fontsize=13, fontweight="bold", pad=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_class_share_pie.png", bbox_inches="tight")
plt.show()


## Plot 03 — Country Contribution per Class


In [ ]:
countries = sorted({a["country"] for a in annotations})
x = np.arange(NUM_CLASSES)
width = 0.8 / max(len(countries), 1)
cmap_c = plt.cm.Set2(np.linspace(0, 1, len(countries)))

fig, ax = plt.subplots(figsize=(12, 5))
for i, country in enumerate(countries):
    src_counts = [
        sum(1 for a in annotations if a["class_id"] == cid and a["country"] == country)
        for cid in range(NUM_CLASSES)
    ]
    offset = (i - len(countries) / 2 + 0.5) * width
    ax.bar(x + offset, src_counts, width * 0.9, label=country,
           color=cmap_c[i], edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=15, ha="right")
ax.set_ylabel("Instance count")
ax.set_title("Country Contribution per Class", fontsize=13, fontweight="bold")
ax.legend(title="Country", fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_country_contribution_per_class.png", bbox_inches="tight")
plt.show()


## Plot 04 — Annotations per Image


In [ ]:
img_ann_count = defaultdict(int)
for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for lf in labels_dir.glob("*.txt"):
            key = country + "_" + split + "_" + lf.stem
            with open(lf) as f:
                img_ann_count[key] = sum(1 for l in f if l.strip())

counts_img = np.array(list(img_ann_count.values()))
counts_nz  = counts_img[counts_img > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.hist(counts_img, bins=40, color="#3498DB", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(counts_img), color="#E74C3C", lw=2, linestyle="--",
           label=f"Mean: {np.mean(counts_img):.2f}")
ax.axvline(np.median(counts_img), color="#F39C12", lw=2, linestyle=":",
           label=f"Median: {np.median(counts_img):.2f}")
ax.set_xlabel("Annotations per image"); ax.set_ylabel("Images")
ax.set_title("Annotations per Image (all)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

ax2 = axes[1]
if len(counts_nz):
    ax2.hist(counts_nz, bins=40, color="#2ECC71", edgecolor="white", alpha=0.85)
    ax2.set_yscale("log")
ax2.set_xlabel("Annotations per image (annotated only)"); ax2.set_ylabel("Count (log)")
ax2.set_title("Annotations per Image (log y)", fontsize=12, fontweight="bold")

fig.suptitle("Annotation Load Distribution", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_annotations_per_image.png", bbox_inches="tight")
plt.show()


## Plot 05 — BBox Width Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cid, name in enumerate(CLASS_NAMES):
    w = arrays[cid].get("w", np.array([]))
    if not len(w): continue
    ax.hist(w, bins=80, alpha=0.5, label=name, color=CLASS_COLORS[cid], edgecolor="none")
    ax.axvline(np.mean(w), color=CLASS_COLORS[cid], linestyle="--", linewidth=1.2)
ax.set_xlabel("BBox width (normalised 0–1)"); ax.set_ylabel("Count")
ax.set_title("BBox Width Distribution per Class", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, title="Class", ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_bbox_width_distribution.png", bbox_inches="tight")
plt.show()


## Plot 06 — BBox Height Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cid, name in enumerate(CLASS_NAMES):
    h = arrays[cid].get("h", np.array([]))
    if not len(h): continue
    ax.hist(h, bins=80, alpha=0.5, label=name, color=CLASS_COLORS[cid], edgecolor="none")
    ax.axvline(np.mean(h), color=CLASS_COLORS[cid], linestyle="--", linewidth=1.2)
ax.set_xlabel("BBox height (normalised 0–1)"); ax.set_ylabel("Count")
ax.set_title("BBox Height Distribution per Class", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, title="Class", ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_bbox_height_distribution.png", bbox_inches="tight")
plt.show()


## Plot 07 — BBox Area Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for cid, name in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    if not len(areas): continue
    ax.hist(np.log10(areas + 1e-9), bins=60, alpha=0.5, label=name,
            color=CLASS_COLORS[cid], edgecolor="none")
ax.set_xlabel("log₁₀(bbox area)"); ax.set_ylabel("Count")
ax.set_title("BBox Area (log scale)", fontsize=12, fontweight="bold")
ax.legend(fontsize=8, ncol=2)

ax2 = axes[1]
data_box, labels_box, color_box = [], [], []
for cid, name in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    if len(areas):
        data_box.append(np.log10(areas + 1e-9))
        labels_box.append(name)
        color_box.append(CLASS_COLORS[cid])
if data_box:
    bp = ax2.boxplot(data_box, patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
    for patch, c in zip(bp["boxes"], color_box):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax2.set_xticklabels(labels_box, rotation=30, ha="right", fontsize=8)
    ax2.set_ylabel("log₁₀(area)")
    ax2.set_title("BBox Area Box Plot per Class", fontsize=12, fontweight="bold")

fig.suptitle("Bounding Box Area Analysis", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_bbox_area_distribution.png", bbox_inches="tight")
plt.show()


## Plot 08 — Aspect Ratio Violin


In [ ]:
valid_ar = [(name, arrays[cid]["aspect"])
            for cid, name in enumerate(CLASS_NAMES)
            if len(arrays[cid].get("aspect", []))]

if valid_ar:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    names_v = [v[0] for v in valid_ar]
    data_v  = [v[1] for v in valid_ar]
    cids_v  = [CLASS_NAMES.index(n) for n in names_v]

    ax = axes[0]
    parts = ax.violinplot(data_v, showmedians=True, showextrema=False)
    for body, cid in zip(parts["bodies"], cids_v):
        body.set_facecolor(CLASS_COLORS[cid]); body.set_alpha(0.7)
    parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(2)
    ax.set_xticks(range(1, len(names_v)+1))
    ax.set_xticklabels(names_v, rotation=30, ha="right", fontsize=8)
    ax.axhline(1.0, color="gray", linestyle="--", alpha=0.6, label="square (w=h)")
    ax.set_ylabel("w / h"); ax.set_title("Aspect Ratio Violin", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylim(0, min(np.percentile(np.concatenate(data_v), 99) * 1.3, 15))

    ax2 = axes[1]
    means = [np.mean(d) for d in data_v]
    stds  = [np.std(d)  for d in data_v]
    xp = np.arange(len(names_v))
    ax2.bar(xp, means, color=[CLASS_COLORS[i] for i in cids_v], alpha=0.8, edgecolor="white")
    ax2.errorbar(xp, means, yerr=stds, fmt="none", color="black", capsize=5, linewidth=1.5)
    ax2.axhline(1.0, color="gray", linestyle="--", alpha=0.6)
    ax2.set_xticks(xp); ax2.set_xticklabels(names_v, rotation=30, ha="right", fontsize=8)
    ax2.set_ylabel("Mean aspect ratio ± std")
    ax2.set_title("Mean Aspect Ratio per Class", fontsize=12, fontweight="bold")

    fig.suptitle("Aspect Ratio Analysis", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "08_aspect_ratio_violin.png", bbox_inches="tight")
    plt.show()


## Plot 09 — Spatial Density Heatmap


In [ ]:
GRID = 32
centres = {cid: ([], []) for cid in range(NUM_CLASSES)}
for ann in annotations:
    centres[ann["class_id"]][0].append(ann["cx"])
    centres[ann["class_id"]][1].append(ann["cy"])

active = [cid for cid in range(NUM_CLASSES) if centres[cid][0]]
cols = min(len(active), 4)
rows = (len(active) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4.5*rows))
axes_flat = np.array(axes).flatten() if len(active) > 1 else [axes]

for ax, cid in zip(axes_flat, active):
    cx = np.array(centres[cid][0])
    cy = np.array(centres[cid][1])
    heatmap, _, _ = np.histogram2d(cx, cy, bins=GRID, range=[[0,1],[0,1]])
    im = ax.imshow(heatmap.T, origin="upper", aspect="equal",
                   cmap="YlOrRd", interpolation="bilinear", extent=[0,1,1,0])
    plt.colorbar(im, ax=ax, shrink=0.8, label="Count")
    ax.set_title(CLASS_NAMES[cid], fontsize=10, fontweight="bold")
    ax.set_xlabel("cx (norm)"); ax.set_ylabel("cy (norm)")
    ax.tick_params(labelsize=7)

for ax in axes_flat[len(active):]:
    ax.axis("off")

fig.suptitle("Spatial Density of BBox Centres (normalised)", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_spatial_density_heatmap.png", bbox_inches="tight")
plt.show()


## Plot 10 — COCO Size Categories
YOLO coordinates are normalised (0–1). We scale to 640×640 for COCO thresholds.


In [ ]:
IMG_SIZE = 640
SMALL_MAX  = (32  / IMG_SIZE) ** 2
MEDIUM_MAX = (96  / IMG_SIZE) ** 2

active_ids   = [cid for cid in range(NUM_CLASSES) if len(arrays[cid].get("area", []))]
active_names = [CLASS_NAMES[cid] for cid in active_ids]
small_c, medium_c, large_c = [], [], []
for cid in active_ids:
    a = arrays[cid]["area"]
    small_c.append(int(np.sum(a < SMALL_MAX)))
    medium_c.append(int(np.sum((a >= SMALL_MAX) & (a < MEDIUM_MAX))))
    large_c.append(int(np.sum(a >= MEDIUM_MAX)))

xp = np.arange(len(active_names)); bw = 0.25
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for i, (cnts, lbl, clr) in enumerate(zip(
        [small_c, medium_c, large_c],
        ["Small (<32px)", "Medium (32–96px)", "Large (>96px)"],
        ["#3498DB", "#F39C12", "#E74C3C"])):
    ax.bar(xp + (i-1)*bw, cnts, bw, label=lbl, color=clr, alpha=0.8, edgecolor="white")
ax.set_xticks(xp); ax.set_xticklabels(active_names, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Count"); ax.set_title("COCO Size Categories (grouped)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

ax2 = axes[1]
totals = [s+m+l for s,m,l in zip(small_c, medium_c, large_c)]
s_p = [s/t*100 if t else 0 for s,t in zip(small_c, totals)]
m_p = [m/t*100 if t else 0 for m,t in zip(medium_c, totals)]
l_p = [l/t*100 if t else 0 for l,t in zip(large_c, totals)]
ax2.bar(xp, s_p, color="#3498DB", alpha=0.8, label="Small", edgecolor="white")
ax2.bar(xp, m_p, bottom=s_p, color="#F39C12", alpha=0.8, label="Medium", edgecolor="white")
ax2.bar(xp, l_p, bottom=[s+m for s,m in zip(s_p,m_p)],
        color="#E74C3C", alpha=0.8, label="Large", edgecolor="white")
ax2.set_xticks(xp); ax2.set_xticklabels(active_names, rotation=30, ha="right", fontsize=8)
ax2.set_ylabel("%"); ax2.set_ylim(0, 100)
ax2.set_title("COCO Size Categories (normalised)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)

fig.suptitle("COCO Size Category Analysis  (scaled to 640×640)", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_coco_size_categories.png", bbox_inches="tight")
plt.show()


## Plot 11 — Class Co-occurrence Heatmap


In [ ]:
img_classes = defaultdict(set)
for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for lf in labels_dir.glob("*.txt"):
            key = f"{country}_{split}_{lf.stem}"
            with open(lf) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cid = int(parts[0])
                        if cid < NUM_CLASSES:
                            img_classes[key].add(cid)

matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for classes in img_classes.values():
    for i in classes:
        for j in classes:
            matrix[i, j] += 1

diag = matrix.diagonal().astype(float)
diag[diag == 0] = 1
norm_matrix = matrix / diag[:, None]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[0], cbar_kws={"label": "Count"})
axes[0].set_title("Co-occurrence (raw)", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)

sns.heatmap(norm_matrix, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
            cbar_kws={"label": "P(col|row)"})
axes[1].set_title("Co-occurrence P(col|row)", fontsize=12, fontweight="bold")
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=8)

fig.suptitle("Class Co-occurrence per Image", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_class_cooccurrence_heatmap.png", bbox_inches="tight")
plt.show()


## Plot 12 — Compactness vs Area Scatter


In [ ]:
MAX_PTS = 2000
np.random.seed(42)
fig, ax = plt.subplots(figsize=(10, 5))
for cid, name in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    comp  = arrays[cid].get("compactness", np.array([]))
    if not len(areas): continue
    n = len(areas)
    if n > MAX_PTS:
        idx = np.random.choice(n, MAX_PTS, replace=False)
        areas = areas[idx]; comp = comp[idx]
    ax.scatter(np.log10(areas + 1e-9), comp,
               c=[CLASS_COLORS[cid]], alpha=0.35, s=10,
               label=f"{name} (n={n:,})", edgecolors="none")

ax.set_xlabel("log₁₀(bbox area)"); ax.set_ylabel("Compactness (4π·A/P²)")
ax.set_title("BBox Compactness vs Area\n(1=circle, ~0.05=elongated crack)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, markerscale=2, title="Class", ncol=2)
ax.axhline(0.785, color="gray", linestyle=":", linewidth=1, alpha=0.5)
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_compactness_vs_area_scatter.png", bbox_inches="tight")
plt.show()


## Summary


In [ ]:
plots = sorted(OUTPUT_DIR.glob("*.png"))
print(f"\n✅ {len(plots)} plots saved to {OUTPUT_DIR}")
for p in plots:
    print(f"  {p.name}")
